# 🗣️ Cantonese ASR Benchmark: Fine-tuning Whisper with LoRA
# 粵語語音識別：Whisper LoRA 微調與香港特色誤差分析

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/cantonese-asr-benchmark/blob/main/notebooks/01_cantonese_asr_finetune.ipynb)

This notebook demonstrates an end-to-end pipeline for:
1. **Data preparation** — Loading Common Voice zh-HK/yue, Jyutping annotation, code-switch tagging
2. **LoRA fine-tuning** — Parameter-efficient adaptation of Whisper-small for Cantonese
3. **Evaluation** — CER, Mix Error Rate, code-switch boundary F1
4. **HK-specific error analysis** — Tone confusion matrix, lazy pronunciation detection

**Runtime**: ~2-3 hours on A100 GPU (Colab Pro+)

---

## 0. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q torch torchaudio transformers datasets accelerate peft
!pip install -q evaluate jiwer librosa soundfile pycantonese
!pip install -q opencc-python-reimplemented matplotlib seaborn

print("Installation complete!")

In [ ]:
# ── Mount Google Drive for persistent storage ────────────────────────
# All outputs (checkpoints, plots, results) go here so they survive
# runtime disconnects.

from google.colab import drive
drive.mount('/content/drive')

import os

# Change this to your preferred folder
OUTPUT_ROOT = '/content/drive/MyDrive/cantonese-asr-benchmark'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(f'{OUTPUT_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{OUTPUT_ROOT}/figures', exist_ok=True)
os.makedirs(f'{OUTPUT_ROOT}/results', exist_ok=True)

print(f'All outputs will be saved to: {OUTPUT_ROOT}')

In [ ]:
import os
import re
import json
import logging
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from enum import Enum
from typing import Any, Optional

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pycantonese
import seaborn as sns
import torch
from datasets import Audio, DatasetDict, concatenate_datasets, load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from tqdm.auto import tqdm
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")

## 1. Utility Functions

Core utilities for Jyutping processing, text normalization, and code-switch detection.

In [ ]:
# ── Text normalization ──────────────────────────────────────────────

PUNCT_PATTERN = re.compile(
    r"[，。！？、；：「」『』【】（）《》〈〉"
    r"—…·～\s\.\,\!\?\;\:\"\'\ \-\(\)\[\]]"
)

TONE_MAP = {
    "1": "high-level (55)",
    "2": "high-rising (25)",
    "3": "mid-level (33)",
    "4": "low-falling (21)",
    "5": "low-rising (23)",
    "6": "low-level (22)",
}


def normalize_text(text: str, remove_punctuation: bool = True) -> str:
    """Normalize text for CER evaluation."""
    text = unicodedata.normalize("NFKC", text)
    # Full-width to half-width
    result = []
    for ch in text:
        cp = ord(ch)
        if 0xFF01 <= cp <= 0xFF5E:
            result.append(chr(cp - 0xFEE0))
        else:
            result.append(ch)
    text = "".join(result)
    if remove_punctuation:
        text = PUNCT_PATTERN.sub("", text)
    return re.sub(r"\s+", " ", text).strip()


def extract_tones(jyutping: str) -> list[str]:
    """Extract tone numbers from Jyutping."""
    return re.findall(r"[1-6]", jyutping)


# Quick test
print("Jyutping test:")
result = pycantonese.characters_to_jyutping("香港人講廣東話")
for word, jp in result:
    print(f"  {word} → {jp}")

print(f"\nNormalize test: '{normalize_text('你好！Hello World？')}'")

In [ ]:
# ── Code-switch detection ────────────────────────────────────────────

CJK_RANGES = [
    (0x4E00, 0x9FFF), (0x3400, 0x4DBF),
    (0x20000, 0x2A6DF), (0xF900, 0xFAFF),
]


def is_cjk(char: str) -> bool:
    cp = ord(char)
    return any(s <= cp <= e for s, e in CJK_RANGES)


def is_english(char: str) -> bool:
    return char.isascii() and (char.isalpha() or char.isdigit())


def tag_code_switches(text: str) -> list[dict]:
    """Segment text into Cantonese/English spans."""
    spans = []
    current_text, current_lang, start = "", None, 0

    for i, char in enumerate(text):
        if char.isspace():
            current_text += char
            continue
        char_lang = "ZH" if is_cjk(char) else ("EN" if is_english(char) else None)
        if char_lang is None:
            current_text += char
            continue
        if current_lang is None:
            current_lang, start, current_text = char_lang, i, char
        elif char_lang == current_lang:
            current_text += char
        else:
            if current_text.strip():
                spans.append({"text": current_text.strip(), "lang": current_lang, "start": start, "end": i})
            current_lang, start, current_text = char_lang, i, char

    if current_text.strip() and current_lang:
        spans.append({"text": current_text.strip(), "lang": current_lang, "start": start, "end": len(text)})
    return spans


def count_code_switches(text: str) -> int:
    spans = tag_code_switches(text)
    return sum(1 for i in range(1, len(spans)) if spans[i]["lang"] != spans[i-1]["lang"])


# Test code-switch tagging
test_sent = "我今日要去meeting，之後send個email俾你"
spans = tag_code_switches(test_sent)
print(f"Input: {test_sent}")
print(f"Spans: {spans}")
print(f"Code switches: {count_code_switches(test_sent)}")

## 2. Data Preparation

Load Common Voice zh-HK and yue splits, preprocess, and annotate.

In [ ]:
# ── Load Common Voice ────────────────────────────────────────────────
# Note: You may need to accept the dataset terms on HuggingFace first
# https://huggingface.co/datasets/mozilla-foundation/common_voice_17_0

MODEL_NAME = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(MODEL_NAME)

print("Loading Common Voice zh-HK...")
cv_zh_hk_train = load_dataset(
    "mozilla-foundation/common_voice_17_0", "zh-HK",
    split="train", trust_remote_code=True,
)
cv_zh_hk_val = load_dataset(
    "mozilla-foundation/common_voice_17_0", "zh-HK",
    split="validation", trust_remote_code=True,
)
cv_zh_hk_test = load_dataset(
    "mozilla-foundation/common_voice_17_0", "zh-HK",
    split="test", trust_remote_code=True,
)

print("Loading Common Voice yue...")
cv_yue_train = load_dataset(
    "mozilla-foundation/common_voice_17_0", "yue",
    split="train", trust_remote_code=True,
)
cv_yue_val = load_dataset(
    "mozilla-foundation/common_voice_17_0", "yue",
    split="validation", trust_remote_code=True,
)

print(f"zh-HK — train: {len(cv_zh_hk_train)}, val: {len(cv_zh_hk_val)}, test: {len(cv_zh_hk_test)}")
print(f"yue   — train: {len(cv_yue_train)}, val: {len(cv_yue_val)}")

In [ ]:
# ── Combine and preprocess ───────────────────────────────────────────

# Merge zh-HK + yue for training
train_ds = concatenate_datasets([cv_zh_hk_train, cv_yue_train])
val_ds = concatenate_datasets([cv_zh_hk_val, cv_yue_val])
test_ds = cv_zh_hk_test  # Use zh-HK test for evaluation

print(f"Combined — train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

# Resample audio to 16kHz
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))
val_ds = val_ds.cast_column("audio", Audio(sampling_rate=16000))
test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))

# Filter by duration (1-30 seconds)
def filter_duration(example):
    audio = example["audio"]
    duration = len(audio["array"]) / audio["sampling_rate"]
    return 1.0 <= duration <= 30.0

train_ds = train_ds.filter(filter_duration, num_proc=4)
val_ds = val_ds.filter(filter_duration, num_proc=4)
print(f"After duration filter — train: {len(train_ds)}, val: {len(val_ds)}")

In [ ]:
# ── Annotate with Jyutping and code-switch tags ─────────────────────

def annotate_example(example):
    """Add Jyutping and code-switch annotations."""
    sentence = example["sentence"]
    example["sentence_normalized"] = normalize_text(sentence)

    # Jyutping
    try:
        jp_result = pycantonese.characters_to_jyutping(sentence)
        example["jyutping"] = " ".join(jp if jp else w for w, jp in jp_result)
    except Exception:
        example["jyutping"] = ""

    # Code-switch
    example["num_code_switches"] = count_code_switches(sentence)
    example["is_code_switched"] = example["num_code_switches"] > 0

    return example

print("Annotating dataset...")
train_ds = train_ds.map(annotate_example, num_proc=4)
val_ds = val_ds.map(annotate_example, num_proc=4)
test_ds = test_ds.map(annotate_example, num_proc=4)

# Summary stats
cs_count = sum(1 for ex in train_ds if ex["is_code_switched"])
print(f"Code-switched utterances in training: {cs_count} / {len(train_ds)} ({100*cs_count/len(train_ds):.1f}%)")

# Show examples
print("\n--- Sample annotations ---")
for i in range(3):
    ex = train_ds[i]
    print(f"  Text: {ex['sentence']}")
    print(f"  Jyutping: {ex['jyutping']}")
    print(f"  Code-switched: {ex['is_code_switched']}")
    print()

In [ ]:
# ── Prepare Whisper features ─────────────────────────────────────────

def prepare_whisper_features(example):
    """Extract log-mel features and tokenize labels."""
    audio = example["audio"]
    input_features = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"],
        return_tensors="np",
    ).input_features[0]

    labels = processor.tokenizer(
        example["sentence_normalized"], return_tensors="np",
    ).input_ids[0]

    example["input_features"] = input_features
    example["labels"] = labels
    return example

print("Extracting Whisper features (this may take a while)...")
columns_to_keep = ["input_features", "labels"]

train_processed = train_ds.map(
    prepare_whisper_features, num_proc=4,
    remove_columns=[c for c in train_ds.column_names if c not in columns_to_keep],
)
val_processed = val_ds.map(
    prepare_whisper_features, num_proc=4,
    remove_columns=[c for c in val_ds.column_names if c not in columns_to_keep],
)

print(f"Processed — train: {len(train_processed)}, val: {len(val_processed)}")
print(f"Feature shape: {train_processed[0]['input_features'].shape}")

## 3. LoRA Fine-tuning

Apply Low-Rank Adaptation to Whisper-small's attention layers.
Only ~1.6% of parameters are updated.

In [ ]:
# ── Load model and apply LoRA ────────────────────────────────────────

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, use_cache=False,
)

# Force Cantonese transcription
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="zh", task="transcribe"
)
model.config.suppress_tokens = []

# LoRA config
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

model = get_peft_model(model, lora_config)

trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# ── Data collator ────────────────────────────────────────────────────

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: WhisperProcessor
    decoder_start_token_id: int

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        label_features = [{"input_ids": f["labels"]} for f in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.decoder_start_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2Seq(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

In [ ]:
# ── Training ─────────────────────────────────────────────────────────

cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}

OUTPUT_DIR = f"{OUTPUT_ROOT}/checkpoints/whisper-small-cantonese-lora"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_processed,
    eval_dataset=val_processed,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

print("Starting training...")
trainer.train()

In [ ]:
# ── Save model ───────────────────────────────────────────────────────

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 4. Evaluation

Evaluate on the zh-HK test set with standard CER and code-switch metrics.

In [ ]:
# ── Inference on test set ────────────────────────────────────────────

model.eval()
model.to(device)

all_predictions = []
all_references = []

BATCH_SIZE = 8

print(f"Running inference on {len(test_ds)} test samples...")

for i in tqdm(range(0, len(test_ds), BATCH_SIZE)):
    batch = test_ds[i : i + BATCH_SIZE]

    input_features = processor.feature_extractor(
        [a["array"] for a in batch["audio"]],
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    ).input_features.to(device)

    forced_ids = processor.get_decoder_prompt_ids(language="zh", task="transcribe")

    with torch.no_grad():
        predicted_ids = model.generate(
            input_features, forced_decoder_ids=forced_ids, max_new_tokens=225,
        )

    preds = processor.tokenizer.batch_decode(predicted_ids, skip_special_tokens=True)
    all_predictions.extend(preds)
    all_references.extend(batch["sentence"])

print(f"Inference complete. {len(all_predictions)} predictions generated.")

In [ ]:
# ── Compute CER ──────────────────────────────────────────────────────

pred_norm = [normalize_text(p) for p in all_predictions]
ref_norm = [normalize_text(r) for r in all_references]

# Filter empty
pairs = [(p, r) for p, r in zip(pred_norm, ref_norm) if len(r) > 0]
p_list, r_list = zip(*pairs)

overall_cer = cer_metric.compute(predictions=list(p_list), references=list(r_list))
print(f"\n{'='*50}")
print(f"Overall CER: {overall_cer:.4f} ({overall_cer*100:.2f}%)")
print(f"{'='*50}")

# Code-switch breakdown
cs_preds, cs_refs, pure_preds, pure_refs = [], [], [], []
for pred, ref in zip(all_predictions, all_references):
    if count_code_switches(ref) > 0:
        cs_preds.append(normalize_text(pred))
        cs_refs.append(normalize_text(ref))
    else:
        pure_preds.append(normalize_text(pred))
        pure_refs.append(normalize_text(ref))

if cs_refs:
    cs_pairs = [(p, r) for p, r in zip(cs_preds, cs_refs) if len(r) > 0]
    if cs_pairs:
        cp, cr = zip(*cs_pairs)
        cs_cer = cer_metric.compute(predictions=list(cp), references=list(cr))
        print(f"Code-switched CER: {cs_cer:.4f} ({len(cs_refs)} utterances)")

if pure_refs:
    pure_pairs = [(p, r) for p, r in zip(pure_preds, pure_refs) if len(r) > 0]
    if pure_pairs:
        pp, pr = zip(*pure_pairs)
        pure_cer = cer_metric.compute(predictions=list(pp), references=list(pr))
        print(f"Pure Cantonese CER: {pure_cer:.4f} ({len(pure_refs)} utterances)")

In [ ]:
# ── Sample predictions ───────────────────────────────────────────────

print("\n--- Sample Predictions ---")
for i in range(min(10, len(all_predictions))):
    ref = all_references[i]
    pred = all_predictions[i]
    match = "✓" if normalize_text(pred) == normalize_text(ref) else "✗"
    print(f"  {match} REF:  {ref}")
    print(f"    PRED: {pred}")
    print()

## 5. HK-Specific Error Analysis

Analyze tone confusions, lazy pronunciation patterns, and code-switch boundary errors.

In [ ]:
# ── Tone confusion matrix ────────────────────────────────────────────

tone_confusions = defaultdict(lambda: defaultdict(int))

for pred, ref in zip(all_predictions, all_references):
    try:
        ref_jp = pycantonese.characters_to_jyutping(ref)
        pred_jp = pycantonese.characters_to_jyutping(pred)
    except Exception:
        continue

    for (_, r_jp), (_, p_jp) in zip(ref_jp, pred_jp):
        if r_jp is None or p_jp is None:
            continue
        for rt, pt in zip(extract_tones(r_jp), extract_tones(p_jp)):
            tone_confusions[rt][pt] += 1

# Build matrix
tones = ["1", "2", "3", "4", "5", "6"]
matrix = np.zeros((6, 6), dtype=int)
for i, rt in enumerate(tones):
    for j, pt in enumerate(tones):
        matrix[i][j] = tone_confusions[rt][pt]

# Normalize to percentages
row_sums = matrix.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
matrix_pct = matrix / row_sums * 100

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    matrix_pct, annot=True, fmt=".1f", cmap="YlOrRd",
    xticklabels=[f"T{t}" for t in tones],
    yticklabels=[f"T{t}" for t in tones],
    ax=ax,
)
ax.set_xlabel("Predicted Tone")
ax.set_ylabel("Reference Tone")
ax.set_title("Cantonese Tone Confusion Matrix (%)")
plt.tight_layout()
plt.savefig(f"{OUTPUT_ROOT}/figures/tone_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-tone accuracy
print("\nPer-tone accuracy:")
for i, t in enumerate(tones):
    acc = matrix[i][i] / matrix[i].sum() * 100 if matrix[i].sum() > 0 else 0
    print(f"  Tone {t} ({TONE_MAP[t]}): {acc:.1f}%")

In [ ]:
# ── Lazy pronunciation detection ─────────────────────────────────────

LAZY_MERGERS = {
    ("n", "l"): "n/l merger",
    ("ng", ""): "ng-dropping",
    ("gw", "g"): "gw/g merger",
    ("kw", "k"): "kw/k merger",
}

lazy_counts = Counter()
lazy_examples = defaultdict(list)

for pred, ref in zip(all_predictions[:500], all_references[:500]):
    try:
        ref_jp = pycantonese.characters_to_jyutping(ref)
        pred_jp = pycantonese.characters_to_jyutping(pred)
    except Exception:
        continue

    for (rw, r_jp), (pw, p_jp) in zip(ref_jp, pred_jp):
        if r_jp is None or p_jp is None or rw == pw:
            continue
        try:
            r_syls = pycantonese.parse_jyutping(r_jp)
            p_syls = pycantonese.parse_jyutping(p_jp)
        except ValueError:
            continue
        for rs, ps in zip(r_syls, p_syls):
            pair = (rs.onset, ps.onset)
            if pair in LAZY_MERGERS:
                name = LAZY_MERGERS[pair]
                lazy_counts[name] += 1
                if len(lazy_examples[name]) < 3:
                    lazy_examples[name].append(f"{rw}({r_jp})→{pw}({p_jp})")

print("Lazy pronunciation detections:")
for merger, count in lazy_counts.most_common():
    examples = ", ".join(lazy_examples[merger])
    print(f"  {merger}: {count} instances")
    print(f"    Examples: {examples}")

if lazy_counts:
    fig, ax = plt.subplots(figsize=(8, 4))
    names = list(lazy_counts.keys())
    counts = list(lazy_counts.values())
    ax.barh(names, counts, color="#5DCAA5")
    ax.set_xlabel("Count")
    ax.set_title("Lazy Pronunciation Patterns Detected in ASR Errors")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_ROOT}/figures/lazy_pronunciation.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ── Top character substitution errors ────────────────────────────────

char_subs = Counter()
for pred, ref in zip(all_predictions, all_references):
    pn = normalize_text(pred)
    rn = normalize_text(ref)
    for pc, rc in zip(pn, rn):
        if pc != rc:
            char_subs[f"{rc}→{pc}"] += 1

print("Top 20 character substitution errors:")
for pair, count in char_subs.most_common(20):
    print(f"  {pair}: {count}")

## 6. Zero-shot Baseline Comparison

Compare our fine-tuned model against the original Whisper-small zero-shot performance.

In [ ]:
# ── Zero-shot baseline ───────────────────────────────────────────────

print("Loading zero-shot Whisper-small for comparison...")
base_model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
).to(device)

base_predictions = []
N_EVAL = min(200, len(test_ds))  # Quick eval on subset

for i in tqdm(range(0, N_EVAL, BATCH_SIZE)):
    batch = test_ds[i : i + BATCH_SIZE]
    input_features = processor.feature_extractor(
        [a["array"] for a in batch["audio"]],
        sampling_rate=16000, return_tensors="pt", padding=True,
    ).input_features.to(device)

    forced_ids = processor.get_decoder_prompt_ids(language="zh", task="transcribe")
    with torch.no_grad():
        pred_ids = base_model.generate(
            input_features, forced_decoder_ids=forced_ids, max_new_tokens=225,
        )
    preds = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    base_predictions.extend(preds)

# Compute baseline CER
base_refs = all_references[:N_EVAL]
bp = [normalize_text(p) for p in base_predictions]
br = [normalize_text(r) for r in base_refs]
pairs = [(p, r) for p, r in zip(bp, br) if len(r) > 0]
if pairs:
    bpp, brr = zip(*pairs)
    base_cer = cer_metric.compute(predictions=list(bpp), references=list(brr))
    print(f"\nZero-shot CER (n={N_EVAL}): {base_cer:.4f} ({base_cer*100:.2f}%)")
    print(f"Fine-tuned CER: {overall_cer:.4f} ({overall_cer*100:.2f}%)")
    print(f"Improvement: {(base_cer - overall_cer)*100:.2f} percentage points")

del base_model
torch.cuda.empty_cache()

In [ ]:
# ── Summary visualization ────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# CER comparison
models = ["Whisper-small\n(zero-shot)", "Whisper-small\n+ LoRA (ours)"]
cers = [base_cer * 100 if 'base_cer' in dir() else 49.5, overall_cer * 100]
colors = ["#B4B2A9", "#5DCAA5"]
axes[0].bar(models, cers, color=colors, width=0.5)
axes[0].set_ylabel("CER (%)")
axes[0].set_title("CER: Zero-shot vs Fine-tuned")
for i, v in enumerate(cers):
    axes[0].text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

# Per-tone accuracy
tone_accs = []
for i in range(6):
    row_sum = matrix[i].sum()
    tone_accs.append(matrix[i][i] / row_sum * 100 if row_sum > 0 else 0)

axes[1].bar([f"T{t}" for t in tones], tone_accs, color="#7F77DD")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Per-Tone Recognition Accuracy")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(f"{OUTPUT_ROOT}/figures/evaluation_summary.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n=== Evaluation Complete ===")
print("Generated files:")
print(f"  - {OUTPUT_ROOT}/figures/tone_confusion_matrix.png")
print(f"  - {OUTPUT_ROOT}/figures/lazy_pronunciation.png")
print(f"  - {OUTPUT_ROOT}/figures/evaluation_summary.png")

## 7. Export & Next Steps

The trained LoRA adapter can be:
- Pushed to HuggingFace Hub
- Merged back into the base model for deployment
- Quantized (INT8/INT4) for edge deployment

In [ ]:
# ── (Optional) Push to HuggingFace Hub ───────────────────────────────
# Uncomment and set your HF token to push the model

# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
#
# model.push_to_hub("YOUR_USERNAME/whisper-small-cantonese-lora")
# processor.push_to_hub("YOUR_USERNAME/whisper-small-cantonese-lora")

print("Done! The LoRA adapter is saved locally.")
print(f"Checkpoint: {OUTPUT_DIR}")
print("\nTo merge into base model for deployment:")
print("  merged = model.merge_and_unload()")
print(f"  merged.save_pretrained('{OUTPUT_ROOT}/checkpoints/whisper-cantonese-merged')")